## Experimental AI Telegram Chatbot (Gemini 2.5 Flash-Lite)

**Key Features:**
* **Long Polling (timeout=30)**: Keeps the bot responsive in real-time without overwhelming the Telegram API.
* **Message Offsets**: Uses unique IDs to "bookmark" progress, preventing the bot from responding to the same message multiple times.
* **Markdown Formatting**: Enables the bot to display bold text, bullet points, and code snippets generated by the AI.

To get a Telegram Bot HTTP API token, search for "@BotFather" in the Telegram app, start a chat, and use the /newbot command. Follow the instructions to name your bot, and BotFather will provide the token needed to authenticate your bot.

**Steps to Get a New Token**
1.   Open Telegram and search for @BotFather (ensure it has a blue checkmark).
2.   Click Start and send the message /newbot.
3.   Choose a name for your bot.
4.   Create a unique username for your bot, which must end in "bot" (e.g., MyTest_bot).
5.   BotFather will send a message with the API token (a long string of letters and numbers).


*Note: Ensure you have your token from BotFather and have added it to your 'Secrets' (🔑) as `telegram`.*

In [30]:
import requests
import time
from google.colab import userdata
import google.generativeai as genai

# --- 1. CONFIGURATION ---
# Securely retrieve your secrets from the Colab "Keys" menu
GEMINI_KEY = userdata.get('GEMINI_API_KEY')
TOKEN = userdata.get('telegram')
BASE_URL = f'https://api.telegram.org/bot{TOKEN}/'

In [31]:
# --- 2. GEMINI AI SETUP (2.5 Flash-Lite) ---
genai.configure(api_key=GEMINI_KEY)

# Initializing Gemini 2.5 Flash-Lite for high-speed, professional responses
model = genai.GenerativeModel('gemini-2.5-flash-lite')

def get_ai_response(prompt):
    """
    Interfaces with Gemini 2.5 Flash-Lite.
    Includes a system instruction to maintain a professional persona.
    """
    system_instruction = "You are a helpful assistant. Be concise."
    try:
        # Generate the AI response based on user input
        response = model.generate_content(f"{system_instruction} User asks: {prompt}")
        return response.text
    except Exception as e:
        print(f"AI Error: {e}")
        return "I'm having a brief technical moment. Please try again!"

In [32]:
# --- 3. TELEGRAM SEND LOGIC ---
def send_telegram_msg(chat_id, text):
    """
    Delivers the AI's response back to the user's phone.
    Uses Markdown to ensure professional formatting.
    """
    url = f"{BASE_URL}sendMessage"
    payload = {
        "chat_id": chat_id,
        "text": text,
        "parse_mode": "Markdown"
    }
    # Using POST is more robust for sending AI-generated text
    requests.post(url, json=payload)

In [33]:
# --- 4. MAIN CHAT LOOP ---
last_update_id = None
print("AI Chatbot is now active and listening for messages...")

while True:
    url = f"{BASE_URL}getUpdates?timeout=30"
    if last_update_id:
        url += f"&offset={last_update_id}"

    try:
        res = requests.get(url).json()

        if "result" in res:
            for update in res["result"]:
                if "message" in update and "text" in update["message"]:
                    chat_id = update["message"]["chat"]["id"]
                    user_text = update["message"]["text"]

                    # 1. Show the user's question in Colab
                    print(f"\n[USER]: {user_text}")

                    # 2. Get the AI response
                    ai_reply = get_ai_response(user_text)

                    # 3. Show the AI's answer in Colab
                    print(f"[AI]: {ai_reply}")

                    # 4. Send to Telegram
                    send_telegram_msg(chat_id, ai_reply)

                    last_update_id = update["update_id"] + 1

    except Exception as e:
        print(f"Connection Error: {e}")
        time.sleep(5)


AI Chatbot is now active and listening for messages...

[USER]: Explain Reinforcement Learning
[AI]: Reinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by taking actions in an environment to maximize a cumulative reward. It learns through trial and error, receiving positive rewards for good actions and negative rewards (penalties) for bad ones.


KeyboardInterrupt: 

### Implementation Notes:
* **Persona Control**: The `system_instruction` in the code ensures the AI doesn't drift into casual or irrelevant topics.
* **Error Resilience**: The `try/except` blocks are critical for keeping the bot online if the API service is temporarily unavailable.
* **Data Integrity**: Unlike the simple loop in the experimental file, this version uses `last_update_id` to ensure no student message is ever processed twice or missed.